# DBpedia SPARQL notebook

Run SPARQL queries against the [DBpedia SPARQL endpoint](https://dbpedia.org/sparql) and show results with Polars.

| | |
|---|---|
| **Endpoint** | `https://dbpedia.org/sparql` |
| **UI** | https://dbpedia.org/sparql |
| **Project** | Smart-grid knowledge queries |

Replace the sample query below with your own logic when ready.

In [34]:
from typing import Any

import polars as pl
from IPython.display import display
from SPARQLWrapper import JSON, SPARQLWrapper

DBPEDIA_ENDPOINT = "https://dbpedia.org/sparql"

# Wider notebook display (full column text, no "…" truncation)
pl.Config.set_tbl_width_chars(1000)
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(-1)


def run_sparql(query: str, endpoint: str = DBPEDIA_ENDPOINT) -> pl.DataFrame:
    """Execute a SPARQL SELECT query and return bindings as a Polars DataFrame."""
    client = SPARQLWrapper(endpoint)
    client.setQuery(query)
    client.setReturnFormat(JSON)

    payload: dict[str, Any] = client.query().convert()
    bindings = payload.get("results", {}).get("bindings", [])
    rows = [{key: value.get("value").split("/")[-1] for key, value in row.items()} for row in bindings]
    return pl.DataFrame(rows) if rows else pl.DataFrame()


def show_df(df: pl.DataFrame) -> pl.DataFrame:
    """Print the full table without truncating long cell values."""
    display(df)
    return df

Give me first five movie names and creator names in tuples where propery is creator

In [35]:
SAMPLE_QUERY = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT  ?movie ?creator_name WHERE {
  ?movie dbo:creator ?creator_name
}
LIMIT 5
"""

df = run_sparql(SAMPLE_QUERY)
show_df(df)

movie,creator_name
str,str
"""Love_Is_Blind:_Sweden""","""Chris_Coelen"""
"""Love_Is_Blind:_Germany""","""Chris_Coelen"""
"""Love_Is_Blind:_UK""","""Chris_Coelen"""
"""Love_Is_Blind_(American_TV_series)""","""Chris_Coelen"""
"""Love_Is_Blind:_France""","""Chris_Coelen"""


movie,creator_name
str,str
"""Love_Is_Blind:_Sweden""","""Chris_Coelen"""
"""Love_Is_Blind:_Germany""","""Chris_Coelen"""
"""Love_Is_Blind:_UK""","""Chris_Coelen"""
"""Love_Is_Blind_(American_TV_series)""","""Chris_Coelen"""
"""Love_Is_Blind:_France""","""Chris_Coelen"""


Query: Give me all movies and creators whose names starts with "a"
Limit first 5 

In [36]:
SAMPLE_QUERY_TWO = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT  ?movie ?creator_name WHERE {
  ?movie dbo:creator ?creator_name . 
  ?creator_name rdfs:label ?creator_name_label .
  FILTER (STRSTARTS(LCASE(?creator_name_label), "a"))
}
LIMIT 5
"""

df = run_sparql(SAMPLE_QUERY_TWO).unique()
show_df(df)

movie,creator_name
str,str
"""Sandman_(DC_Comics)""","""Allen_Bert_Christman"""
"""Fairy_Godmother_(Shrek)""","""Andrew_Adamson"""


movie,creator_name
str,str
"""Sandman_(DC_Comics)""","""Allen_Bert_Christman"""
"""Fairy_Godmother_(Shrek)""","""Andrew_Adamson"""


Give me publication for publication and publisher nam

In [37]:
SAMPLE_QUERY_THREE = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT  ?publication ?publisher_name WHERE {
  ?publication dbo:publisher ?publisher_name. 
}
LIMIT 5
"""

df = run_sparql(SAMPLE_QUERY_THREE).unique()
show_df(df)

publication,publisher_name
str,str
"""Wavelength_(game)""","""CMYK_(game_publisher)"""
"""Thank_You_(Falettinme_Be_Mice_Elf_Agin)_(book)""","""Auwa_Books"""
"""Alice_Knott__Void_Corporation__1""","""Archway_Editions"""
"""Molly_(memoir)""","""Archway_Editions"""
"""Arthur_-_The_Dog_Who_Crossed_the_Jungle_to_Find_a_Home""",""""""


publication,publisher_name
str,str
"""Wavelength_(game)""","""CMYK_(game_publisher)"""
"""Thank_You_(Falettinme_Be_Mice_Elf_Agin)_(book)""","""Auwa_Books"""
"""Alice_Knott__Void_Corporation__1""","""Archway_Editions"""
"""Molly_(memoir)""","""Archway_Editions"""
"""Arthur_-_The_Dog_Who_Crossed_the_Jungle_to_Find_a_Home""",""""""


Give me publication which starts which publisher name starts with "s"

In [38]:
SAMPLE_QUERY_FOUR = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT  ?publication ?publisher_name_label WHERE {
  ?publication dbo:publisher ?publisher_name. 
  ?publisher_name rdfs:label ?publisher_name_label .
  FILTER (STRSTARTS(LCASE(?publisher_name_label), "s"))
}
LIMIT 5
"""

df = run_sparql(SAMPLE_QUERY_FOUR)
show_df(df)

publication,publisher_name_label
str,str
"""The_World_Development_Report_2011""","""Světová banka"""
"""The_Sunken_Billions""","""Světová banka"""
"""Zombicide""","""San Benediktoren Ordena"""
"""Fiol's_Octoechos""","""Schweipolt Fiol"""
"""Fiol's_Octoechos""","""Schweipolt Fiol"""


publication,publisher_name_label
str,str
"""The_World_Development_Report_2011""","""Světová banka"""
"""The_Sunken_Billions""","""Světová banka"""
"""Zombicide""","""San Benediktoren Ordena"""
"""Fiol's_Octoechos""","""Schweipolt Fiol"""
"""Fiol's_Octoechos""","""Schweipolt Fiol"""


Give me sources 

In [45]:
SAMPLE_QUERY_FIVE = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT  ?thing ?source_name WHERE {
  ?thing dbo:license ?source_name . 
}
LIMIT 5
"""

df = run_sparql(SAMPLE_QUERY_FIVE)
show_df(df)

thing,source_name
str,str
"""Gemma_(language_model)""","""terms%7CGemma"""
"""Intel_Unison""","""W:en:Proprietary_software"""
"""vatinoxan""","""vatinoxan"""
"""cilgavimab""","""cilgavimab"""
"""MSN_Messenger""","""Bundled_software"""


thing,source_name
str,str
"""Gemma_(language_model)""","""terms%7CGemma"""
"""Intel_Unison""","""W:en:Proprietary_software"""
"""vatinoxan""","""vatinoxan"""
"""cilgavimab""","""cilgavimab"""
"""MSN_Messenger""","""Bundled_software"""
